# Day 3 - Gold, Delta and reconciliation

| Task | Cells |
| --- | --- |
| 1. Fact grain, conformed dimensions, surrogate keys, meter/substation joins | this cell, cell 3 |
| 2. Daily kWh, peak load, missing interval rate | cell 4 |
| 3. Silver-to-Gold record, metric and financial reconciliation | cell 5 |
| 4. Production MERGE (full recomputation, guarded) and same-input rerun (no double counting) | cells 4, 6, 7 |
| 5. Delta history, time travel, Spark plan, skew | cells 8, 9 |

### Task 1 - model decisions

| Item | Decision |
| --- | --- |
| Fact grain | `fact_daily_meter`: one row per `meter_id` + `reading_date` (UTC calendar day; source timestamps end in `Z`) |
| Eligible meter-days | taken from Silver (`03_silver_data`): one row per meter-day in `silver_meter_expected_intervals`; expected intervals = Silver expected 5-min slots (observation window = first to last valid slot) |
| Valid readings | `silver_smart_meter_readings` = rows that passed every ERROR rule (45 sentinels in `quarantine_smart_meter_readings`, 49 duplicates removed); pre-installation readings kept and counted in `pre_install_readings` |
| Time zone | UTC in both 03 and 04 (`spark.sql.session.timeZone`); `reading_date` = UTC calendar day of `interval_start_ts` |
| Substation assignment | current meter master (`dim_meter.substation_id`) for both daily kWh and coincident peak; readings with a different `substation_id` (Silver RDG-025) reported by TC07-08. One substation per meter, no effective dates -> Type 1, no SCD2 claim |
| `active_flag` | BOOLEAN in Silver and Gold; `upsert()` refuses any source/target type mismatch, so the STRING column in the live `dim_substation` needs an approved migration first |
| Production refresh | full recomputation from Silver + differential MERGE (insert new, update changed, delete meter-days no longer in Silver), preceded by a read-only guard (Silver inputs present and from one 03 run, unique and complete source, exact insert/update/delete preview, `MAX_PROD_DELETE_ROWS` threshold). The `_ingested_at` watermark path is not CDC: it cannot see removals or window changes and runs only on the test table |
| Tests | TC03 and late-arrival run on `test_fact_daily_meter` / `test_gold_watermark` only; production fact and watermark versions are checked unchanged |
| Conformed dimensions | `dim_meter` (NK `meter_id`), `dim_substation` (NK `substation_id`), `dim_date` (`date_sk` = yyyymmdd) |
| Surrogate keys | stable BIGINT: existing keys reused, new business keys get max(sk)+n, never regenerated (not necessarily contiguous) |
| Joins | fact.meter_sk -> dim_meter; fact.substation_sk -> dim_substation; fact.date_sk -> dim_date; dim_meter.substation_id -> dim_substation.substation_id (natural). All 1:N to the fact |
| Second fact | `fact_daily_substation`: one row per substation per day; peak = max over 5-min slots of summed meter kW (coincident), not sum of meter peaks |
| Financial | blocked: meter tariff codes (T01-T03) do not match lookup codes (TARIFF09000xx); no assumed mapping |

In [0]:
# ============================================================
# Configuration and shared helpers
# ============================================================
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

CAT, SLV, GLD = "capstone9_smartmeter", "smart_grid_silver", "smart_grid_gold"
S = lambda t: f"{CAT}.{SLV}.{t}"      # Silver table name
G = lambda t: f"{CAT}.{GLD}.{t}"      # Gold table name
FACT = G("fact_daily_meter")

SLOT = F.timestamp_seconds(F.floor(F.unix_timestamp("interval_start_ts") / 300) * 300)   # 5-min slot of a reading

spark.conf.set("spark.sql.session.timeZone", "UTC")   # source timestamps end in 'Z' -> reading_date = UTC day
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.{GLD}")


def valid_readings():
    """Silver readings from 03: only rows that passed every ERROR rule (quarantine and duplicates already removed)."""
    return spark.table(S("silver_smart_meter_readings"))


# Production delete threshold: the MERGE aborts if MORE target rows would be deleted.
# 0 = no production delete without review. Raise it only after inspecting the printed rows to be deleted.
MAX_PROD_DELETE_ROWS = 0


def preview_merge(src, table, keys, keep=()):
    """Read-only: exact insert / update / delete counts that upsert(..., delete_missing=True) would apply.
    Uses the same update rule as upsert (null-safe difference on every non-key column except gold_updated_at)."""
    keys = list(keys)
    if not spark.catalog.tableExists(table):
        return {"insert": src.count(), "update": 0, "delete": 0, "deleted_keys": None}
    t = spark.table(table)
    compare = [c for c in src.columns if c not in set(keys) | set(keep) | {"gold_updated_at"}]
    differs = None
    for c in compare:
        e = ~F.col(f"s.{c}").eqNullSafe(F.col(f"t.{c}"))
        differs = e if differs is None else differs | e
    deleted_keys = t.select(*keys).join(src.select(*keys), keys, "left_anti")   # target rows absent from source
    return {
        "insert": src.select(*keys).join(t.select(*keys), keys, "left_anti").count(),
        "update": src.alias("s").join(t.alias("t"), keys).where(differs).count() if compare else 0,
        "delete": deleted_keys.count(),
        "deleted_keys": deleted_keys,
    }


def check_deletes(p, table):
    """Print the proposed changes and abort (before any write) if deletes exceed MAX_PROD_DELETE_ROWS."""
    print(f"{table} proposed MERGE: insert={p['insert']} update={p['update']} delete={p['delete']} "
          f"(delete threshold {MAX_PROD_DELETE_ROWS})")
    if p["delete"] > MAX_PROD_DELETE_ROWS:
        display(p["deleted_keys"].limit(100))
        raise RuntimeError(f"{table}: {p['delete']} rows would be deleted (> {MAX_PROD_DELETE_ROWS}); "
                           "nothing written - review the rows above before raising MAX_PROD_DELETE_ROWS")


def upsert(src, table, keys, keep=(), delete_missing=False):
    """MERGE on business keys: insert new rows, update only rows whose values changed.
    delete_missing=True (only when src is the COMPLETE recomputed set) also deletes target rows
    that no longer exist in src. An identical rerun inserts, updates and deletes nothing (idempotent)."""
    if not spark.catalog.tableExists(table):
        src.limit(0).write.saveAsTable(table)                     # empty table with the source schema

    # Schema guard: never let MERGE silently cast a source column to a different target type
    tgt = dict(spark.table(table).dtypes)
    mismatched = {c: f"source {t} vs target {tgt[c]}" for c, t in src.dtypes if c in tgt and tgt[c] != t}
    new_cols = [c for c in src.columns if c not in tgt]
    if mismatched or new_cols:
        raise ValueError(f"{table}: schema mismatch {mismatched}, new columns {new_cols} - "
                         "apply an approved schema migration first; nothing written")

    fixed = set(keys) | set(keep)                                 # columns never updated
    compare = [c for c in src.columns if c not in fixed | {"gold_updated_at"}]

    merge = (DeltaTable.forName(spark, table).alias("t")
        .merge(src.alias("s"), " AND ".join(f"t.{k} = s.{k}" for k in keys))
        .whenMatchedUpdate(
            condition=" OR ".join(f"NOT (t.{c} <=> s.{c})" for c in compare),
            set={c: f"s.{c}" for c in src.columns if c not in fixed})
        .whenNotMatchedInsertAll())
    if delete_missing:
        merge = merge.whenNotMatchedBySourceDelete()              # meter-days removed from Silver
    merge.execute()

    return DeltaTable.forName(spark, table).history(1).select("version", "operationMetrics").first()


def upsert_dim(src, table, nk, sk):
    """Surrogate-key policy: existing business keys keep their key, new keys get max(sk) + n.
    Keys are never regenerated."""
    if not spark.catalog.tableExists(table):
        src.select(F.lit(None).cast("bigint").alias(sk), "*").limit(0).write.saveAsTable(table)

    current = spark.table(table).select(nk, sk)
    start = current.agg(F.coalesce(F.max(sk), F.lit(0))).first()[0]
    new_keys = (src.join(current, nk, "left_anti")
                   .withColumn(sk, (F.lit(start) + F.row_number().over(Window.orderBy(nk))).cast("bigint")))   # small dim: 1 partition is fine

    return upsert(src.join(current, nk).unionByName(new_keys), table, [nk], keep=[sk])


def log(name, m):
    """One-line MERGE summary from the Delta operation metrics."""
    om = m.operationMetrics
    print(f"{name:<22} v{m.version}  inserted={om['numTargetRowsInserted']}  updated={om['numTargetRowsUpdated']}")

In [0]:
# ============================================================
# Task 1 - Conformed dimensions: dim_substation, dim_meter, dim_date
# ============================================================

def latest(table, key):
    """One current row per business key (latest Silver ingestion)."""
    w = Window.partitionBy(key).orderBy(F.col("_ingested_at").desc())
    return spark.table(S(table)).withColumn("_rn", F.row_number().over(w)).where("_rn = 1")


# Substation dimension (business key substation_id)
# active_flag: BOOLEAN in Silver (03 maps Y/N -> true/false, other values quarantined by SUB-008) and in Gold.
# The live dim_substation still has STRING: upsert() refuses the write until the approved migration is applied.
dim_substation = latest("silver_substation_master", "substation_id").select(
    "substation_id", "substation_name", "region", "capacity_mva", "voltage_level_kv",
    "commissioned_date", "operator", "active_flag", "latitude", "longitude",
    F.current_timestamp().alias("gold_updated_at"))

# Meter dimension (business key meter_id; substation_id = natural link to dim_substation)
dim_meter = latest("silver_meter_master", "meter_id").select(
    "meter_id", "substation_id", "meter_type", "installation_date", "service_region",
    "meter_status", "customer_class", "rated_voltage", "manufacturer", "tariff_code",
    F.current_timestamp().alias("gold_updated_at"))

# Date dimension covering the Silver reporting period (date_sk = yyyymmdd)
d = F.col("full_date")
dim_date = (valid_readings()
    .agg(F.sequence(F.min(F.to_date("interval_start_ts")), F.max(F.to_date("interval_start_ts"))).alias("ds"))
    .select(F.explode("ds").alias("full_date"))
    .select(F.date_format(d, "yyyyMMdd").cast("int").alias("date_sk"),
            "full_date",
            F.dayofmonth(d).alias("day"),
            F.month(d).alias("month"),
            F.date_format(d, "MMMM").alias("month_name"),
            F.quarter(d).alias("quarter"),
            F.year(d).alias("year"),
            F.date_format(d, "EEEE").alias("day_of_week"),
            F.current_timestamp().alias("gold_updated_at")))

log("dim_substation", upsert_dim(dim_substation, G("dim_substation"), "substation_id", "substation_sk"))
log("dim_meter", upsert_dim(dim_meter, G("dim_meter"), "meter_id", "meter_sk"))
log("dim_date", upsert(dim_date, G("dim_date"), ["date_sk"]))

In [0]:
# ============================================================
# Task 2 - Daily Gold metrics  |  Task 4 - incremental MERGE
# ============================================================

def daily_meter_metrics(readings, meters):
    """Recalculate every Silver meter-day of the given meters (never adds to old totals)."""
    dm = spark.table(G("dim_meter")).select("meter_id", "meter_sk", "substation_id", "installation_date")
    r = (readings.join(meters, "meter_id")
                 .join(dm.select("meter_id", "installation_date"), "meter_id", "left")
                 .withColumn("reading_date", F.to_date("interval_start_ts"))
                 .withColumn("slot_ts", SLOT))

    # Daily kWh, peak load (5-min kWh x 12 = average kW) and reported intervals per meter-day
    agg = r.groupBy("meter_id", "reading_date").agg(
        F.round(F.sum("energy_kwh"), 4).alias("daily_kwh"),
        F.round(F.max(F.col("energy_kwh") * 12), 4).alias("peak_load_kw"),
        F.countDistinct("slot_ts").alias("reported_intervals"),
        F.sum((F.col("reading_date") < F.col("installation_date")).cast("int")).alias("pre_install_readings"))

    # Eligible meter-days and expected intervals come from Silver (incl. days with no readings)
    expected = (spark.table(S("silver_meter_expected_intervals")).join(meters, "meter_id")
                     .groupBy("meter_id", "reading_date")
                     .agg(F.count("*").cast("int").alias("expected_intervals")))

    return (expected.join(agg, ["meter_id", "reading_date"], "left")
        .fillna(0, ["daily_kwh", "reported_intervals", "pre_install_readings"])
        # Surrogate-key lookups: meter -> substation through the meter master
        .join(dm.drop("installation_date"), "meter_id", "left")
        .join(spark.table(G("dim_substation")).select("substation_id", "substation_sk"), "substation_id", "left")
        .join(spark.table(G("dim_date")).select("date_sk", F.col("full_date").alias("reading_date")), "reading_date", "left")
        # Missing intervals and missing interval rate (%)
        .withColumn("missing_intervals", F.col("expected_intervals") - F.col("reported_intervals"))
        .withColumn("missing_interval_rate", F.round(F.col("missing_intervals") * 100 / F.col("expected_intervals"), 4))
        .select("meter_sk", "substation_sk", "date_sk", "meter_id", "reading_date",
                "daily_kwh", "peak_load_kw", "expected_intervals", "reported_intervals",
                "missing_intervals", "missing_interval_rate", "pre_install_readings",
                F.current_timestamp().alias("gold_updated_at")))


PROD_WM = G("gold_watermark")
FACT_KEYS = ["meter_id", "reading_date"]


def production_guard(src, readings):
    """Read-only checks before ANY write to the production fact. Collects every problem, then raises.
    Expected meter-day population = distinct (meter_id, reading_date) in silver_meter_expected_intervals
    (03 grid: every meter with >= 1 valid reading x every UTC day of the first-to-last valid 5-min slot window)."""
    problems = []

    # 1. Required Silver inputs exist, are non-empty and come from the same 03 run
    exp_t = S("silver_meter_expected_intervals")
    for t in (S("silver_smart_meter_readings"), exp_t):
        if not spark.catalog.tableExists(t):
            raise RuntimeError(f"Production guard: {t} missing - nothing written")
    exp = spark.table(exp_t)
    if readings.limit(1).count() == 0:
        problems.append("silver_smart_meter_readings is empty")
    if exp.limit(1).count() == 0:
        problems.append("silver_meter_expected_intervals is empty")
    run_r = {x[0] for x in readings.select("_silver_run_id").distinct().collect()}
    run_e = {x[0] for x in exp.select("_silver_run_id").distinct().collect()}
    if len(run_r) != 1 or run_r != run_e:
        problems.append(f"Silver inputs from different 03 runs: readings={run_r}, expected={run_e}")

    # 2. Expected population is consistent with the valid readings
    population = exp.select(*FACT_KEYS).distinct()
    reading_days = readings.select("meter_id", F.to_date("interval_start_ts").alias("reading_date")).distinct()
    n_pop = population.count()
    # Rectangular grid: every grid meter must have every grid UTC day (03 builds meters x window days)
    n_grid_meters = population.select("meter_id").distinct().count()
    n_grid_days = population.select("reading_date").distinct().count()
    if n_pop != n_grid_meters * n_grid_days:
        problems.append(f"expected grid not rectangular: {n_pop} meter-days != "
                        f"{n_grid_meters} meters x {n_grid_days} UTC days = {n_grid_meters * n_grid_days}")
    grid_meters_without_readings = (population.select("meter_id").distinct()
                                    .join(readings.select("meter_id").distinct(), "meter_id", "left_anti").count())
    reading_days_outside_grid = reading_days.join(population, FACT_KEYS, "left_anti").count()
    if grid_meters_without_readings:
        problems.append(f"{grid_meters_without_readings} meters in the expected grid have no valid readings")
    if reading_days_outside_grid:
        problems.append(f"{reading_days_outside_grid} meter-days with valid readings are outside the expected grid")

    # 3. Source: non-empty, unique (meter_id, reading_date), complete against the population, keys resolved
    n_src = src.count()
    src_keys = src.select(*FACT_KEYS)
    n_src_keys = src_keys.distinct().count()
    null_keys = src.where(F.col("meter_id").isNull() | F.col("reading_date").isNull()).count()
    missing_in_src = population.join(src_keys, FACT_KEYS, "left_anti").count()
    extra_in_src = src_keys.join(population, FACT_KEYS, "left_anti").count()
    null_sk = src.where(F.col("meter_sk").isNull() | F.col("substation_sk").isNull() | F.col("date_sk").isNull()).count()
    if n_src == 0:
        problems.append("source is empty")
    if n_src != n_src_keys or null_keys:
        problems.append(f"source keys not unique/non-null: rows={n_src}, distinct keys={n_src_keys}, null keys={null_keys}")
    if missing_in_src or extra_in_src:
        problems.append(f"source incomplete: {missing_in_src} population meter-days missing, {extra_in_src} extra")
    if null_sk:
        problems.append(f"{null_sk} source rows without meter_sk / substation_sk / date_sk")

    print(f"Production guard: population={n_pop} meter-days, source rows={n_src}, distinct keys={n_src_keys}, "
          f"missing={missing_in_src}, extra={extra_in_src}, null sk={null_sk}, Silver run={run_r}")
    if problems:
        raise RuntimeError("Production guard failed - nothing written: " + "; ".join(problems))

    # 4. Exact proposed changes; abort if deletes exceed the threshold
    check_deletes(preview_merge(src, FACT, FACT_KEYS), FACT)


def run_gold_fact(full=True, readings=None, target=FACT, wm_table=PROD_WM):
    """full=True (production default): recompute EVERY Silver meter-day and MERGE only the differences -
    insert new, update changed, delete meter-days no longer in Silver. Because 03 rebuilds Silver in full,
    this covers corrections, newly quarantined/removed readings, late arrivals and a changed Silver window.
    full=False: watermark fast path on _ingested_at (NOT CDC) - only meters with newer readings are
    recalculated; it cannot see removals or window changes, so it is used only on the isolated test table.
    readings= overrides the Silver input and is allowed only for a test target."""
    if target != FACT and wm_table == PROD_WM:
        raise ValueError("Test runs must use their own watermark table")
    if target == FACT and wm_table != PROD_WM:
        raise ValueError(f"Production runs must use exactly {PROD_WM}, got {wm_table}")
    if readings is not None and target == FACT:
        raise ValueError("Synthetic readings are never written to the production fact table")
    r = readings if readings is not None else valid_readings()

    wm = None
    if not full and spark.catalog.tableExists(wm_table):
        last = spark.table(wm_table).orderBy(F.col("run_at").desc()).first()
        wm = last["last_ingested_at"] if last else None

    changed = r if wm is None else r.where(F.col("_ingested_at") > wm)
    src = daily_meter_metrics(r, changed.select("meter_id").distinct())
    if target == FACT:
        if not full:
            raise ValueError("The production fact is refreshed only with full=True")
        production_guard(src, r)            # read-only; raises before any production write
    result = upsert(src, target, FACT_KEYS, delete_missing=wm is None)   # delete only with a complete source

    # Advance the watermark only after the MERGE succeeded (upsert raises on failure)
    new_wm = changed.agg(F.max("_ingested_at")).first()[0] or wm
    (spark.createDataFrame([(target, new_wm, int(result.version))],
                           "table_name string, last_ingested_at timestamp, fact_version long")
          .withColumn("run_at", F.current_timestamp())
          .write.mode("append").saveAsTable(wm_table))
    return result


def build_substation_daily():
    """fact_daily_substation: one row per substation per day.
    Peak = max over 5-min slots of the summed meter load (coincident), not the sum of meter peaks.
    Substation assignment rule (kWh AND peak): current meter master, dim_meter.substation_id.
    The reading's own substation_id is not used; mismatches are reported by TC07-08 (Silver warning RDG-025)."""
    r = (valid_readings().drop("substation_id")
         .join(spark.table(G("dim_meter")).select("meter_id", "substation_id"), "meter_id")
         .withColumn("reading_date", F.to_date("interval_start_ts"))
         .withColumn("slot_ts", SLOT))

    peak = (r.groupBy("substation_id", "reading_date", "slot_ts").agg(F.sum("energy_kwh").alias("kwh"))
             .groupBy("substation_id", "reading_date")
             .agg(F.round(F.max(F.col("kwh") * 12), 4).alias("peak_coincident_load_kw")))

    daily = spark.table(FACT).groupBy("substation_sk", "date_sk", "reading_date").agg(
        F.round(F.sum("daily_kwh"), 4).alias("daily_kwh"),
        F.sum("expected_intervals").alias("expected_intervals"),
        F.sum("reported_intervals").alias("reported_intervals"),
        F.sum("missing_intervals").alias("missing_intervals"),
        F.count("*").alias("eligible_meters"),
        F.sum((F.col("reported_intervals") > 0).cast("int")).alias("active_meters"))

    out = (daily.join(spark.table(G("dim_substation")).select("substation_sk", "substation_id"), "substation_sk")
                .join(peak, ["substation_id", "reading_date"], "left")
                .withColumn("missing_interval_rate", F.round(F.col("missing_intervals") * 100 / F.col("expected_intervals"), 4))
                .withColumn("gold_updated_at", F.current_timestamp()))
    check_deletes(preview_merge(out, G("fact_daily_substation"), ["substation_sk", "date_sk"]),
                  G("fact_daily_substation"))   # read-only; raises before the write
    return upsert(out, G("fact_daily_substation"), ["substation_sk", "date_sk"], delete_missing=True)   # complete rebuild from the fact


# Production load: full recomputation + differential MERGE (the only write to fact_daily_meter / gold_watermark)
log("fact_daily_meter", run_gold_fact(full=True))
log("fact_daily_substation", build_substation_daily())
display(spark.table(FACT).orderBy("meter_id", "reading_date").limit(10))

In [0]:
%sql
-- ============================================================
-- Task 3 - Silver-to-Gold reconciliation: expected (Silver) vs actual (Gold)
-- Silver side = 03 tables and an independent SQL aggregate of Silver readings
-- ============================================================
USE CATALOG capstone9_smartmeter;

CREATE OR REPLACE TABLE smart_grid_gold.gold_reconciliation_results AS
-- Silver sources
WITH v   AS (SELECT meter_id, to_date(interval_start_ts) AS d, energy_kwh
             FROM smart_grid_silver.silver_smart_meter_readings),            -- valid readings only (03)
sde      AS (SELECT * FROM smart_grid_silver.silver_meter_daily_energy),     -- Silver meter-day totals
f        AS (SELECT * FROM smart_grid_gold.fact_daily_meter),
j        AS (SELECT sde.*, f.daily_kwh, f.peak_load_kw AS g_peak, f.expected_intervals AS g_exp,
                    f.reported_intervals AS g_rep, f.missing_intervals AS g_miss, f.missing_interval_rate AS g_rate,
                    f.meter_id IS NULL AS missing_in_gold, sde.meter_id IS NULL AS missing_in_silver
             FROM sde FULL JOIN f ON sde.meter_id = f.meter_id AND sde.reading_date = f.reading_date),
c AS (
  -- Record reconciliation (TC08)
  SELECT 'TC08-01' AS check_id, 'Fact rows = Silver meter-days' AS check_name, (SELECT count(*) FROM sde) AS expected, (SELECT count(*) FROM f) AS actual
  UNION ALL SELECT 'TC08-02', 'Meter-days missing on either side', 0, (SELECT count_if(missing_in_gold OR missing_in_silver) FROM j)
  UNION ALL SELECT 'TC08-03', 'Meter-days with valid readings', (SELECT count_if(reading_count > 0) FROM sde), (SELECT count_if(reported_intervals > 0) FROM f)

  -- Metric reconciliation (TC08)
  UNION ALL SELECT 'TC08-04', 'Total kWh: Silver readings vs Gold', (SELECT round(sum(energy_kwh), 2) FROM v), (SELECT round(sum(daily_kwh), 2) FROM f)
  UNION ALL SELECT 'TC08-05', 'Total kWh: silver_meter_daily_energy vs Gold', (SELECT round(sum(total_kwh), 2) FROM sde), (SELECT round(sum(daily_kwh), 2) FROM f)
  UNION ALL SELECT 'TC08-06', 'Meter-days with kWh difference', 0, (SELECT count_if(abs(coalesce(total_kwh, 0) - coalesce(daily_kwh, 0)) > 0.0001) FROM j)
  UNION ALL SELECT 'TC08-07', 'Meter-days with peak load difference', 0, (SELECT count_if(NOT (round(peak_load_kw, 4) <=> g_peak)) FROM j)
  UNION ALL SELECT 'TC08-08', 'Reported intervals = valid Silver readings', (SELECT count(*) FROM v), (SELECT sum(reported_intervals) FROM f)
  UNION ALL SELECT 'TC08-09', 'Expected intervals: Silver vs Gold', (SELECT sum(expected_intervals) FROM sde), (SELECT sum(expected_intervals) FROM f)
  UNION ALL SELECT 'TC08-10', 'Meter-days with expected/reported/missing difference', 0,
    (SELECT count_if(NOT (expected_intervals <=> g_exp AND reported_intervals <=> g_rep AND missing_intervals <=> g_miss)) FROM j)
  UNION ALL SELECT 'TC08-11', 'Rows where expected <> reported + missing', 0, (SELECT count_if(expected_intervals <> reported_intervals + missing_intervals) FROM f)
  UNION ALL SELECT 'TC08-12', 'Meter-days with missing rate difference (Gold % vs Silver fraction x 100)', 0,
    (SELECT count_if(abs(g_rate - missing_interval_rate * 100) > 0.0001) FROM j)
  UNION ALL SELECT 'TC08-13', 'Substation daily kWh = meter daily kWh', (SELECT round(sum(daily_kwh), 2) FROM f), (SELECT round(sum(daily_kwh), 2) FROM smart_grid_gold.fact_daily_substation)

  -- Dimension joins and key integrity (TC07)
  UNION ALL SELECT 'TC07-01', 'Duplicate meter-day keys', 0, (SELECT count(*) FROM (SELECT meter_id, reading_date FROM f GROUP BY meter_id, reading_date HAVING count(*) > 1))
  UNION ALL SELECT 'TC07-02', 'Null meter_sk / substation_sk / date_sk', 0, (SELECT count_if(meter_sk IS NULL OR substation_sk IS NULL OR date_sk IS NULL) FROM f)
  UNION ALL SELECT 'TC07-03', 'Orphan fact keys (meter + substation + date)', 0,
      (SELECT count(*) FROM f LEFT ANTI JOIN smart_grid_gold.dim_meter USING (meter_sk))
    + (SELECT count(*) FROM f LEFT ANTI JOIN smart_grid_gold.dim_substation USING (substation_sk))
    + (SELECT count(*) FROM f LEFT ANTI JOIN smart_grid_gold.dim_date USING (date_sk))
  UNION ALL SELECT 'TC07-04', 'Fact substation differs from meter master substation', 0,
    (SELECT count(*) FROM f JOIN smart_grid_gold.dim_meter m ON f.meter_sk = m.meter_sk
                            JOIN smart_grid_gold.dim_substation s ON f.substation_sk = s.substation_sk WHERE m.substation_id <> s.substation_id)
  UNION ALL SELECT 'TC07-05', 'dim_meter rows = distinct Silver meter_id', (SELECT count(DISTINCT meter_id) FROM smart_grid_silver.silver_meter_master), (SELECT count(*) FROM smart_grid_gold.dim_meter)
  UNION ALL SELECT 'TC07-06', 'dim_substation rows = distinct Silver substation_id', (SELECT count(DISTINCT substation_id) FROM smart_grid_silver.silver_substation_master), (SELECT count(*) FROM smart_grid_gold.dim_substation)
  UNION ALL SELECT 'TC07-07', 'Duplicate natural or surrogate keys in dimensions', 0,
      (SELECT 2 * count(*) - count(DISTINCT meter_id) - count(DISTINCT meter_sk) FROM smart_grid_gold.dim_meter)
    + (SELECT 2 * count(*) - count(DISTINCT substation_id) - count(DISTINCT substation_sk) FROM smart_grid_gold.dim_substation)
    + (SELECT count(*) - count(DISTINCT date_sk) FROM smart_grid_gold.dim_date)

  -- Substation assignment: readings whose own substation_id differs from the meter master (Silver RDG-025).
  -- Gold assigns these by meter master; reported as WARN, not hidden
  UNION ALL SELECT 'TC07-08', 'Silver readings whose substation_id differs from meter master (assigned by meter master)', 0,
    (SELECT count(*) FROM smart_grid_silver.silver_smart_meter_readings r JOIN smart_grid_gold.dim_meter m USING (meter_id)
     WHERE r.substation_id <> m.substation_id)

  -- Financial reconciliation: blocked until the meter-to-tariff mapping is confirmed
  UNION ALL SELECT 'FIN-01', 'Meters whose tariff_code exists in tariff lookup', (SELECT count(*) FROM smart_grid_silver.silver_meter_master),
    (SELECT count(*) FROM smart_grid_silver.silver_meter_master m LEFT SEMI JOIN smart_grid_silver.silver_tariff_lookup t ON m.tariff_code = t.tariff_code)
)
-- Expected vs actual with PASS / FAIL / BLOCKED status
SELECT check_id, check_name, CAST(expected AS DOUBLE) AS expected, CAST(actual AS DOUBLE) AS actual,
       CAST(actual - expected AS DOUBLE) AS difference,
       CASE WHEN check_id = 'FIN-01' THEN 'BLOCKED - pending meter-to-tariff mapping; no revenue calculated'
            WHEN check_id = 'TC07-08' AND actual > 0 THEN 'WARN - substation mismatch reported; meter master used'
            WHEN abs(actual - expected) < 0.005 THEN 'PASS' ELSE 'FAIL' END AS status,
       current_timestamp() AS checked_at
FROM c;

-- Evidence
SELECT * FROM smart_grid_gold.gold_reconciliation_results ORDER BY check_id;

In [0]:
# ============================================================
# Task 4 - Idempotency test (TC03) on an ISOLATED test table
# Production fact_daily_meter and gold_watermark are only READ here (snapshot), never written.
# ============================================================
TEST_FACT = G("test_fact_daily_meter")      # test copy of the fact, rebuilt from Silver
TEST_WM = G("test_gold_watermark")          # test watermark (production watermark untouched)
TEST_LOG = G("gold_idempotency_test")       # PASS/FAIL log (append)
assert TEST_FACT != FACT and TEST_WM != PROD_WM


def reset_test_tables():
    """Drop only this test's own tables (guard: name must start with test_)."""
    for t in (TEST_FACT, TEST_WM):
        assert t.split(".")[-1].startswith("test_") and t not in (FACT, PROD_WM), t
        spark.sql(f"DROP TABLE IF EXISTS {t}")


def snapshot(table):
    """Fact-level totals used for before/after comparisons."""
    return spark.sql(f"""
        SELECT count(*)                               AS fact_rows,
               count(DISTINCT meter_id, reading_date) AS meter_day_keys,
               round(sum(daily_kwh), 4)               AS total_kwh,
               sum(reported_intervals)                AS reported_intervals,
               sum(missing_intervals)                 AS missing_intervals
        FROM {table}""").first().asDict()


def prod_state():
    """Production fingerprint: latest Delta version of the fact and the watermark."""
    return (DeltaTable.forName(spark, FACT).history(1).first().version,
            DeltaTable.forName(spark, PROD_WM).history(1).first().version)


def record(test_id, rows):
    """Append this test's PASS/FAIL rows and report the overall result."""
    df = (spark.createDataFrame(rows, "test_id string, metric string, run_1 double, run_2 double, passed boolean")
               .withColumn("tested_at", F.current_timestamp()))
    df.write.mode("append").saveAsTable(TEST_LOG)
    display(df)
    ok = all(r[4] for r in rows)
    print(f"{test_id}: {'PASS' if ok else 'FAIL'}")
    return ok


prod_before = prod_state()
reset_test_tables()
rows = []
try:
    run_gold_fact(full=True, target=TEST_FACT, wm_table=TEST_WM)                # build the test copy
    run1 = snapshot(TEST_FACT)
    m_replay = run_gold_fact(full=True, target=TEST_FACT, wm_table=TEST_WM)     # identical input again
    m_incr = run_gold_fact(full=False, target=TEST_FACT, wm_table=TEST_WM)      # watermark: nothing new
    run2 = snapshot(TEST_FACT)
    rows += [("TC03", k, float(run1[k]), float(run2[k]), run1[k] == run2[k]) for k in run1]
    for name, m in [("replay", m_replay), ("incremental", m_incr)]:
        for op in ("Inserted", "Updated", "Deleted"):
            n = int(m.operationMetrics.get(f"numTargetRows{op}", 0))
            rows.append(("TC03", f"{name} MERGE rows {op.lower()}", None, float(n), n == 0))
except Exception as e:
    rows.append(("TC03", f"error {type(e).__name__}: {str(e)[:200]}", None, None, False))

rows.append(("TC03", "production fact and watermark unchanged", None, None, prod_state() == prod_before))
tc03_ok = record("TC03", rows)

In [0]:
# ============================================================
# Task 4 - Late-arriving reading on the TEST table only
# The late reading exists only in memory; Silver, fact_daily_meter and gold_watermark are not written.
# ============================================================
prod_before = prod_state()
rows, v_late, v_fix = [], None, None
try:
    if not spark.catalog.tableExists(TEST_FACT):                                  # TC03 cell failed early
        run_gold_fact(full=True, target=TEST_FACT, wm_table=TEST_WM)
    baseline = snapshot(TEST_FACT)
    base = valid_readings()

    # Meter-day with exactly one reading outside the 00:00 slot (after the first day, so 00:00 is an expected slot)
    first_day = base.agg(F.min(F.to_date("interval_start_ts"))).first()[0]
    pick = (base.withColumn("reading_date", F.to_date("interval_start_ts"))
                .groupBy("meter_id", "reading_date")
                .agg(F.count("*").alias("n"), F.min("interval_start_ts").alias("ts"))
                .where((F.col("n") == 1) & (F.hour("ts") >= 1) & (F.col("reading_date") > F.lit(first_day)))
                .orderBy("meter_id", "reading_date")
                .first())

    # Copy that reading into the free 00:00 slot with 1.0 kWh and a new ingestion time
    late = (base.where((F.col("meter_id") == pick.meter_id) & (F.col("interval_start_ts") == pick.ts))
                .withColumn("reading_id", F.lit("LATE_TEST_0001"))
                .withColumn("interval_start_ts", F.to_timestamp(F.lit(str(pick.reading_date))))
                .withColumn("energy_kwh", F.lit(1.0))
                .withColumn("_ingested_at", F.current_timestamp()))
    key = f"meter_id = '{pick.meter_id}' AND reading_date = '{pick.reading_date}'"

    # Watermark run on the test table: only the late meter is recalculated
    before = spark.table(TEST_FACT).where(key).first()
    m_late = run_gold_fact(full=False, readings=base.unionByName(late), target=TEST_FACT, wm_table=TEST_WM)
    v_late = m_late.version
    after = spark.table(TEST_FACT).where(key).first()
    om = m_late.operationMetrics
    rows += [
        ("LATE", "daily_kwh +1.0 on the late meter-day", before.daily_kwh, after.daily_kwh,
         abs(after.daily_kwh - before.daily_kwh - 1.0) < 1e-9),
        ("LATE", "reported_intervals +1", float(before.reported_intervals), float(after.reported_intervals),
         after.reported_intervals == before.reported_intervals + 1),
        ("LATE", "only one meter-day updated, none inserted/deleted", None, float(om["numTargetRowsUpdated"]),
         int(om["numTargetRowsUpdated"]) == 1 and int(om["numTargetRowsInserted"]) == 0
         and int(om.get("numTargetRowsDeleted", 0)) == 0),
    ]

    # Reprocess real Silver -> test copy back to the Silver state
    v_fix = run_gold_fact(full=True, target=TEST_FACT, wm_table=TEST_WM).version
    rows.append(("LATE", "test copy reverted to Silver state", None, None, snapshot(TEST_FACT) == baseline))
except Exception as e:
    rows.append(("LATE", f"error {type(e).__name__}: {str(e)[:200]}", None, None, False))

rows.append(("LATE", "production fact and watermark unchanged", None, None, prod_state() == prod_before))
late_ok = record("LATE", rows)
assert tc03_ok and late_ok, f"Gold tests failed: TC03={tc03_ok}, LATE={late_ok}"

In [0]:
# ============================================================
# Task 5 - Delta history, time travel and table detail (read-only)
# ============================================================

# Production fact: every write (create, MERGEs, no-op reruns)
display(spark.sql(f"DESCRIBE HISTORY {FACT}")
             .select("version", "timestamp", "operation", "operationMetrics"))


def at_version(v):
    """The late-test meter-day in the TEST table as it was at Delta version v."""
    return spark.sql(f"""
        SELECT {v} AS version, meter_id, reading_date, daily_kwh, peak_load_kw,
               reported_intervals, missing_intervals, missing_interval_rate
        FROM {TEST_FACT} VERSION AS OF {v}
        WHERE {key}""")


# Time travel on the test table: before, during and after the late-arriving reading
if v_late is not None and v_fix is not None:
    display(at_version(v_late - 1).unionByName(at_version(v_late)).unionByName(at_version(v_fix)))
else:
    print("Late-arrival test did not complete - no time-travel evidence")

# Format, location, file count, size and partitioning (none: small data, high-cardinality meter_id)
display(spark.sql(f"DESCRIBE DETAIL {FACT}")
             .select("format", "location", "numFiles", "sizeInBytes", "partitionColumns", "properties"))

In [0]:
# ============================================================
# Task 5 - Spark execution plan and skew analysis
# ============================================================

# Plan of the daily fact calculation: Exchange = shuffle, BroadcastHashJoin = dimension lookups
daily_meter_metrics(valid_readings(), valid_readings().select("meter_id").distinct()).explain("formatted")

# Key distribution: max_to_avg close to 1 = no skew (salting/repartitioning only if skew is proven)
display(spark.sql(f"""
    SELECT level,
           count(*)                  AS keys,
           min(n)                    AS min_rows,
           round(avg(n), 1)          AS avg_rows,
           max(n)                    AS max_rows,
           round(max(n) / avg(n), 2) AS max_to_avg
    FROM (
        SELECT 'readings per substation' AS level, count(*) AS n
        FROM {S('silver_smart_meter_readings')} GROUP BY substation_id
        UNION ALL
        SELECT 'readings per meter', count(*)
        FROM {S('silver_smart_meter_readings')} GROUP BY meter_id
        UNION ALL
        SELECT 'meters per substation', count(*)
        FROM {S('silver_meter_master')} GROUP BY substation_id
    )
    GROUP BY level"""))